# 💬 Текст, пара 2. Векторизация: TF-IDF и классификатор твитов

> Сначала: **Файл → Сохранить копию на Диске**.

План:
1. Вспоминаем мешок слов.
2. **TF** — как часто слово в этом твите.
3. **IDF** — насколько слово редкое во всех твитах.
4. **TF-IDF** своими руками и через sklearn.
5. **Учим модель** угадывать настроение твита: мешок слов против TF-IDF.
6. Пишем свой твит — модель угадывает настроение.

In [ ]:
#@title ⚙️ Запустите первой: библиотеки, твиты и проверки { display-mode: "form" }
# Внутри только подготовка. Читать не обязательно :)
import math, os, re, urllib.request
from collections import Counter
import numpy as np
import pandas as pd

if not os.path.exists("tweets.csv"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Scr1ptG/ml_for_3p1/main/06-text/data/tweets.csv", "tweets.csv")


def check(tests, n):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}\n   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание {n} выполнено!")
    return True


BAD = re.compile(r"бля|сук|ху[йеия]|пизд|[её]б|мудак|нах")   # мат в топах слов не показываем


print("✅ Готово. Файл tweets.csv скачан:", os.path.exists("tweets.csv"))


def tokenize(text):
    return re.findall(r"[а-яё]+", text.lower())

In [ ]:
df = pd.read_csv("tweets.csv")
corpus = [
    "кот спит и кот ест",
    "пёс спит и пёс лает",
    "кот и пёс играют",
]
for t in corpus:
    print(tokenize(t))

---
# Часть 1. Чем плох мешок слов

В мешке слов «и» весит столько же, сколько «кот». Но «и» есть **во всех** текстах — оно ничего не говорит о тексте. А «лает» есть только в одном — это важное слово.

**TF-IDF** = **TF** (часто ли слово в этом тексте) × **IDF** (редкое ли оно вообще).

---
# Часть 2. TF — частота слова в тексте

$$TF(слово, текст) = \frac{\text{сколько раз слово в тексте}}{\text{сколько всего слов в тексте}}$$

«кот спит и кот ест»: слов 5, «кот» 2 раза → TF = 2/5 = 0.4.

### ✏️ Задание 1. Функция `tf(word, text)`

In [ ]:
def tf(word, text):
    words = tokenize(text)
    return None   # ← words.count(word) / len(words)


In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
if check([
    ('кот в первом тексте', lambda: abs(tf('кот', corpus[0]) - 0.4) < 1e-9, 'words.count(word) / len(words)'),
    ('слова нет — 0', lambda: tf('лает', corpus[0]) == 0, 'count вернёт 0'),
], 1):
    for w in ["кот", "и", "лает"]:
        print(f"TF({w!r}, {corpus[0]!r}) = {tf(w, corpus[0]):.2f}")

<details><summary>💡 Подсказка</summary>

У списка есть метод `.count(x)` — сколько раз x в нём встречается.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return words.count(___) / len(___)
```
</details>

---
# Часть 3. IDF — редкость слова

$$IDF(слово) = \log\frac{\text{сколько всего текстов}}{\text{в скольких текстах есть слово}}$$

* слово во **всех** текстах: log(3/3) = log 1 = **0** — слово бесполезно;
* слово в **одном** тексте: log(3/1) ≈ **1.1** — слово важное.

### ✏️ Задание 2. Функция `df_count(word)`: в скольких текстах `corpus` есть слово

In [ ]:
def df_count(word):
    return None   # ← sum(word in tokenize(t) for t in corpus)


In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
if check([
    ('«и» во всех трёх', lambda: df_count('и') == 3, 'sum(word in tokenize(t) for t in corpus)'),
    ('«лает» в одном', lambda: df_count('лает') == 1, 'Считаем тексты, а не повторы: «кот кот» — это 1 текст'),
], 2):
    for w in ["и", "кот", "спит", "лает"]:
        print(f"{w!r} есть в {df_count(w)} текстах из {len(corpus)}")

<details><summary>💡 Подсказка</summary>

`word in список` даёт True/False, а `sum` считает True.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return sum(word in tokenize(t) for t in ___)
```
</details>

### ✏️ Задание 3. Функция `idf(word)`

In [ ]:
def idf(word):
    return None   # ← math.log(len(corpus) / df_count(word))


In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
if check([
    ('«и» — 0', lambda: abs(idf('и')) < 1e-9, 'log(3 / 3) = 0'),
    ('«лает» — log 3', lambda: abs(idf('лает') - math.log(3)) < 1e-9, 'math.log(len(corpus) / df_count(word))'),
], 3):
    for w in ["и", "кот", "спит", "лает"]:
        print(f"IDF({w!r}) = {idf(w):.2f}")
    print("Чем реже слово, тем больше IDF.")

<details><summary>💡 Подсказка</summary>

`math.log` — натуральный логарифм.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return math.log(len(corpus) / ___(word))
```
</details>

---
# Часть 4. TF-IDF своими руками

### ✏️ Задание 4. Функция `tfidf(word, text)` = TF × IDF

In [ ]:
def tfidf(word, text):
    return None


In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
if check([
    ('«и» обнулилось', lambda: tfidf('и', corpus[0]) == 0, 'tf(word, text) * idf(word)'),
    ('«кот» в первом тексте', lambda: abs(tfidf('кот', corpus[0]) - 0.4 * math.log(1.5)) < 1e-9, 'Перемножьте две функции'),
], 4):
    vocab = sorted({w for t in corpus for w in tokenize(t)})
    table = pd.DataFrame([[round(tfidf(w, t), 2) for w in vocab] for t in corpus], columns=vocab, index=corpus)
    display(table)
    print("«и» — везде 0. «лает», «ест», «играют» — самые большие: они и отличают тексты.")

<details><summary>💡 Подсказка</summary>

Обе функции уже написаны.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
return tf(word, text) * ___(word)
```
</details>

---
# Часть 5. TF-IDF в sklearn

`TfidfVectorizer` делает то же самое, но с двумя поправками:
* к IDF прибавляет 1 (слово из всех текстов получает не 0, а немного);
* делит каждую строку на её длину, чтобы длинные и короткие твиты были сравнимы.

Цифры поэтому чуть другие, но **порядок** тот же: редкие слова важнее частых.

### ✏️ Задание 5. TF-IDF для `corpus` через `TfidfVectorizer`

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tv = TfidfVectorizer(token_pattern=r"[а-яё]+")
T = None   # ← как CountVectorizer: fit_transform

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
if check([
    ('матрица 3 × 7', lambda: T.shape == (3, 7), 'tv.fit_transform(corpus)'),
    ('«и» — меньше всех в первом тексте', lambda: T[0, tv.vocabulary_['и']] < T[0, tv.vocabulary_['ест']], ''),
], 5):
    display(pd.DataFrame(T.toarray().round(2), columns=tv.get_feature_names_out(), index=corpus))

<details><summary>💡 Подсказка</summary>

Те же методы, что у `CountVectorizer`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
T = tv.___(corpus)
```
</details>

---
# Часть 6. Учим модель угадывать настроение

Делим твиты: **80%** — учиться, **20%** — проверка (модель их не видит).

Модель — **логистическая регрессия**. Это один нейрон: у каждого слова есть **вес**, модель складывает веса слов твита. Сумма больше нуля — позитив, меньше — негатив.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

train_text, test_text, y_train, y_test = train_test_split(df["text"], df["label"], test_size=0.2, random_state=0)
print("учимся на", len(train_text), "твитах, проверяем на", len(test_text))

### ✏️ Задание 6. Векторизуйте твиты мешком слов: `X_train`, `X_test`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
bow_vec = CountVectorizer(token_pattern=r"[а-яё]+")

X_train = None   # ← fit_transform: учим словарь на обучающих
X_test = None         # ← только transform: словарь уже выучен!

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
if check([
    ('обучающие', lambda: X_train.shape[0] == 8000, 'bow_vec.fit_transform(train_text)'),
    ('проверочные — тот же словарь', lambda: X_test.shape == (2000, X_train.shape[1]), 'Для проверки — только transform, без fit'),
], 6):
    print("X_train:", X_train.shape, " X_test:", X_test.shape)

<details><summary>💡 Подсказка</summary>

`fit` — только на обучающих. Иначе модель «подсмотрит» слова из проверки.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
X_train = bow_vec.___(train_text)
X_test = bow_vec.___(test_text)
```
</details>

### ✏️ Задание 7. Обучите модель и посчитайте точность `acc_bow` на проверке

In [ ]:
model_bow = LogisticRegression(max_iter=1000)
# ← model_bow.fit(X_train, y_train)
acc_bow = None   # ← model_bow.score(X_test, y_test)

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
if check([
    ('модель обучена', lambda: hasattr(model_bow, 'coef_'), 'model_bow.fit(X_train, y_train)'),
    ('точность лучше угадывания', lambda: acc_bow is not None and acc_bow > 0.6, 'model_bow.score(X_test, y_test)'),
], 7):
    print(f"Точность (мешок слов): {acc_bow:.1%}   (наугад было бы 50%)")

<details><summary>💡 Подсказка</summary>

`fit(X, y)` — учиться, `score(X, y)` — доля правильных ответов.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
model_bow.fit(X_train, ___)
acc_bow = model_bow.score(X_test, ___)
```
</details>

### ✏️ Задание 8. То же с TF-IDF: `acc_tfidf`

In [ ]:
tfidf_vec = TfidfVectorizer(token_pattern=r"[а-яё]+")
Xt_train = None
Xt_test = None

model_tfidf = LogisticRegression(max_iter=1000)
# ← обучите model_tfidf
acc_tfidf = None

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
if check([
    ('векторы', lambda: Xt_test.shape == (2000, Xt_train.shape[1]), 'fit_transform для train, transform для test'),
    ('точность', lambda: acc_tfidf is not None and acc_tfidf > 0.6, 'fit, потом score'),
], 8):
    print(f"мешок слов: {acc_bow:.1%}   TF-IDF: {acc_tfidf:.1%}")

<details><summary>💡 Подсказка</summary>

Всё как в двух прошлых заданиях, только `tfidf_vec` и `model_tfidf`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
Xt_train = tfidf_vec.fit_transform(train_text)
Xt_test = tfidf_vec.transform(test_text)
model_tfidf.fit(Xt_train, y_train)
acc_tfidf = model_tfidf.score(___, ___)
```
</details>

> 🤔 Почему не 95%? Твиты короткие, а метки ставили по смайликам: «блин, опять дождь :)» — позитив. Даже человек не всегда угадает без смайлика. 65–70% на таких данных — честный результат.

---
# Часть 7. Что выучила модель
У каждого слова есть **вес**. Большой плюс — слово «тянет» к позитиву, большой минус — к негативу.

In [ ]:
words = tfidf_vec.get_feature_names_out()
w = model_tfidf.coef_[0]          # веса слов; положительные — к классу model_tfidf.classes_[1]
print("класс с плюсом:", model_tfidf.classes_[1])
order = [i for i in np.argsort(w) if not BAD.search(words[i])]   # BAD — фильтр мата
print("😞 тянут к негативу:", ", ".join(words[order[:15]]))
print("😊 тянут к позитиву:", ", ".join(words[order[-15:]][::-1]))

### ✏️ Задание 9. Напишите свой твит и узнайте, что скажет модель

In [ ]:
my_tweet = None   # ← любой текст в кавычках
answer = None   # ← model_tfidf.predict(tfidf_vec.transform([my_tweet]))[0]

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
if check([
    ('твит написан', lambda: isinstance(my_tweet, str) and len(tokenize(my_tweet)) > 0, 'Строка в кавычках'),
    ('модель ответила', lambda: answer in ('позитив', 'негатив'), 'Сначала transform (в списке!), потом predict'),
], 9):
    print(f"«{my_tweet}» → {answer}")
    print("Уверенность:", dict(zip(model_tfidf.classes_, model_tfidf.predict_proba(tfidf_vec.transform([my_tweet]))[0].round(2).tolist())))

<details><summary>💡 Подсказка</summary>

`transform` ждёт **список** текстов, поэтому `[my_tweet]`. `predict` тоже возвращает список — берём `[0]`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
answer = model_tfidf.predict(tfidf_vec.transform([___]))[0]
```
</details>

### ✏️ Задание 10. Проверьте «не плохо» и «плохо»: различает ли их модель?

In [ ]:
a1 = model_tfidf.predict(tfidf_vec.transform(["все плохо"]))[0]
a2 = None   # ← то же для "все не плохо"

In [ ]:
#@title 🧪 Проверка задания 10 { display-mode: "form" }
if check([
    ('оба ответа есть', lambda: a1 in ('позитив', 'негатив') and a2 in ('позитив', 'негатив'), 'Скопируйте строку a1 и поменяйте текст'),
], 10):
    print("«все плохо» →", a1, "| «все не плохо» →", a2)
    print("Модель видит отдельные слова. «не» и «плохо» по отдельности — оба негативные.")

<details><summary>💡 Подсказка</summary>

Скопируйте первую строку.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
a2 = model_tfidf.predict(tfidf_vec.transform(["все ___ плохо"]))[0]
```
</details>

### 🔬 Эксперименты
* `TfidfVectorizer(token_pattern=r"[а-яё]+", ngram_range=(1, 2))` — словарь из слов **и пар слов** («не плохо» станет одним токеном). Точность выросла?
* `max_features=1000` — оставить только 1000 самых частых слов. Насколько упала точность?

### ✅ Итог
| | Мешок слов | TF-IDF |
|---|---|---|
| что в векторе | сколько раз слово в твите | частота × редкость |
| частые слова («и», «я») | весят много | приглушены |
| порядок слов | теряется | теряется |
| смысл слов | не знает | не знает → **word2vec** в следующей теме |